# Brazilian Films Data Pipeline | ETL com Python e Pandas

## ETL + Data Clearing + Data Quality

##Este projeto tem como objetivo desenvolver um pipeline ETL para tratamento e preparação de uma base de dados sobre produções cinematográficas.

##A partir de uma base em formato ODS, o pipeline realiza etapas de extração, inspeção, seleção de atributos relevantes, tratamento de valores ausentes, padronização de tipos de dados e transformação das informações utilizando Python e Pandas.

#Ingestão de dados

link do documento: https://dados.gov.br/dados/conjuntos-dados/filmes-e-sessoes-da-programadora-brasil

Confirmamos o upload do arquivo

In [ ]:
import os

os.listdir("/content")

['.config', 'filmes.ods', 'sample_data']

In [ ]:
import pandas as pd

In [ ]:
!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=d493f0e4b4351a3f9612ea465d9b235261faebddb0807f0acea54eb4b7395a8e
  Stored in directory: /root/.cache/pip/wheels/8e/cd/9f/979443982946991080916064e4c049b91941be1800825ff74b
Successfully built odfpy


Lemos o arquivo

In [ ]:
filmes = pd.read_excel("/content/filmes.ods", engine="odf")

inspeção: linhas e colunas do *arquivo*

In [ ]:
filmes.shape

(1879, 121)

5 primeiras linhas

In [ ]:
filmes.head()

,codigo,status,data_cadastro,data_alteracao,titulo,titulo_en,sinopse,sinopse_editada,sinopse_en,diretor,...,dist_exclusivo,pat_minc,pat_minc_edital,pat_minc_contrato,desc_classificacao,autor_da_trilha,animador,dublagem,cod_inscrito_pb,infanto_juvenil
0,148,1,2008-03-01 00:00:00,2011-08-17 11:14:10,Amarelo manga,Mango Yellow,"Guiados pela paixão, os personagens de Amarelo...","Guiados pela paixão, os personagens de Amarelo...",NaN,Cl&aacute;udio Assis,...,N,N,BO - Edital de concurso nº 2 de 27 de abril de...,4412000,NaN,Jorge du Peixe e Lúcio Maia,NaN,NaN,1,N
1,149,1,2008-03-01 00:00:00,2011-08-16 19:00:13,A canga,NaN,"Num descampado, no meio de uma lavoura seca, u...","Num descampado, no meio de uma lavoura seca, u...",NaN,Marcus Vilar,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
2,150,1,2008-03-01 00:00:00,2012-09-28 15:41:32,"Batismo de Carmencita, 25 de junho de 1921",NaN,Um dos assuntos de um cinejornal. Cerimônia de...,Um dos assuntos de um cinejornal. Cerimônia de...,NaN,Autoria desconhecida,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
3,151,1,2008-03-01 00:00:00,2012-09-28 15:49:00,Exemplo regenerador,NaN,Um marido farrista deixa a esposa só em casa n...,Um marido farrista deixa a esposa só em casa n...,NaN,Jos&eacute; Medina,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N
4,152,1,2008-03-01 00:00:00,2012-10-09 15:20:31,Brasilianas: Aboio e Cantigas,NaN,O Canto utilizado pelo vaqueiro para reunir a ...,O Canto utilizado pelo vaqueiro para reunir a ...,NaN,Humberto Mauro,...,N,N,NaN,NaN,NaN,NaN,NaN,NaN,1,N


informações

In [ ]:
filmes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1879 entries, 0 to 1878
Columns: 121 entries, codigo to infanto_juvenil
dtypes: float64(3), int64(21), object(97)
memory usage: 1.7+ MB


## Etapa 1: Selecionar apenas o que interessa

In [ ]:
filmes = filmes[
    [
        "codigo",
        "titulo",
        "diretor",
        "pais",
        "ano",
        "duracao",
        "produtora",
        "festivais",
        "premios",
        "data_lancamento"
    ]
]

##Etapa 2: Verificar qualidade

In [ ]:
filmes.isnull().sum()

,0
codigo,0
titulo,0
diretor,0
pais,1
ano,0
duracao,0
produtora,471
festivais,968
premios,1058
data_lancamento,685


## Etapa 3: Tratar os dados

In [ ]:
filmes["produtora"] = filmes["produtora"].fillna("Não informado")
filmes["festivais"] = filmes["festivais"].fillna("Não informado")
filmes["premios"] = filmes["premios"].fillna("Não informado")

conferir duplicatas

In [ ]:
filmes.duplicated().sum()

np.int64(0)

##Etapa 4: Tipagem

ano (deve ser numérico)

In [ ]:
filmes["ano"] = pd.to_numeric(
    filmes["ano"],
    errors="coerce"
)

duração

In [ ]:
filmes["duracao"] = pd.to_numeric(
    filmes["duracao"],
    errors="coerce"
)

data

In [ ]:
filmes["data_lancamento"] = pd.to_datetime(
    filmes["data_lancamento"],
    errors="coerce",
    dayfirst=True
)

## Etapa 5: criar uma coluna nova

classificação de duração

In [ ]:
def classificar_duracao(minutos):
    if minutos < 30:
        return "Curta-metragem"
    elif minutos < 70:
        return "Média-metragem"
    else:
        return "Longa-metragem"

filmes["tipo_duracao"] = filmes["duracao"].apply(classificar_duracao)

## Etapa 6: Camada final

revisamos todo o arquivo

In [ ]:
filmes.head()

,codigo,titulo,diretor,pais,ano,duracao,produtora,festivais,premios,data_lancamento,tipo_duracao
0,148,Amarelo manga,Cl&aacute;udio Assis,Brasil,2003,100,Olhos de Cão Produções Cinematográficas,Não informado,"- Melhor Filme - Fórum do Novo Cinema, concedi...",2003-08-15,Longa-metragem
1,149,A canga,Marcus Vilar,Brasil,2000,12,Não informado,Não informado,-Prêmio Especial do Júri no Festival de Cinema...,NaT,Curta-metragem
2,150,"Batismo de Carmencita, 25 de junho de 1921",Autoria desconhecida,Brasil,1921,2,Não informado,Não informado,Não informado,NaT,Curta-metragem
3,151,Exemplo regenerador,Jos&eacute; Medina,Brasil,1919,7,Não informado,Não informado,Não informado,NaT,Curta-metragem
4,152,Brasilianas: Aboio e Cantigas,Humberto Mauro,Brasil,1954,10,Não informado,Não informado,Não informado,NaT,Curta-metragem


In [ ]:
filmes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1879 entries, 0 to 1878
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   codigo           1879 non-null   int64         
 1   titulo           1879 non-null   object        
 2   diretor          1879 non-null   object        
 3   pais             1878 non-null   object        
 4   ano              1879 non-null   int64         
 5   duracao          1879 non-null   int64         
 6   produtora        1879 non-null   object        
 7   festivais        1879 non-null   object        
 8   premios          1879 non-null   object        
 9   data_lancamento  203 non-null    datetime64[ns]
 10  tipo_duracao     1879 non-null   object        
dtypes: datetime64[ns](1), int64(3), object(7)
memory usage: 161.6+ KB


In [ ]:
filmes.isnull().sum()

,0
codigo,0
titulo,0
diretor,0
pais,1
ano,0
duracao,0
produtora,0
festivais,0
premios,0
data_lancamento,1676


## Exportação

In [ ]:
filmes.to_csv(
    "filmes_tratados.csv",
    index=False,
    encoding="utf-8-sig"
)